# AI text detection using Qwen3-Embedding-0.6B

Uploading model, it has to be already saved locally

In [1]:
from model_upload import load_model_from_cache


model = load_model_from_cache("Qwen/Qwen3-Embedding-0.6B", "/Users/roberto/Università/Deep learning")

/Users/roberto/Università/Deep learning/AI-text-detection-models/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5410.52it/s]


Model uploaded!


## Creating and saving embeddings

In [ ]:
import torch
import numpy as np

# Supponiamo che tu abbia il tuo dataset diviso in liste
train_sentences = ["Frase umana 1...", "Frase AI 1...", ...]
train_labels = [0, 1, ...] # 0 = Human, 1 = AI

print("Generazione embedding di training...")
# model.encode può restituire direttamente tensori PyTorch impostando convert_to_tensor=True
X_train_embeddings = model.encode(train_sentences, convert_to_tensor=True, show_progress_bar=True)
y_train_labels = torch.tensor(train_labels, dtype=torch.long)

# Salva tutto in un unico dizionario permanente sul disco
torch.save({
    'embeddings': X_train_embeddings,
    'labels': y_train_labels
}, 'train_dataset_embeddings.pt')

print("Dataset di training salvato con successo!")

In [3]:
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium."
]
embeddings = model.encode(sentences)
print(embeddings.shape)
similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

(3, 1024)
torch.Size([3, 3])


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# 1. Carica gli embedding pre-calcolati dal disco
train_data = torch.load('train_dataset_embeddings.pt')
X_train = train_data['embeddings']
y_train = train_data['labels']

# 2. Crea il DataLoader di PyTorch per il training a batch
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 3. Definisci la tua rete Fully Connected
# Qwen-0.6B genera embedding, la dimensione di input dipenderà da quella del modello (es. 1536 o 4096)
input_dim = X_train.shape[1] 

class TextClassifier(nn.Module):
    def __init__(self, input_dim):
        super(TextClassifier, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 2) # 2 classi: Human (0) e AI (1)
        )
        
    def forward(self, x):
        return self.fc(x)

model_fc = TextClassifier(input_dim)

# Ora puoi avviare il loop di training (le epoche gireranno in pochissimi millisecondi!)
# for epoch in range(epochs): ...